# 🧠 Bsta AI — FAISS Vector Store Creator

This notebook converts one or more PDF files into a FAISS vector store
that can be loaded directly by the Streamlit app (`ibt.py`).

**Usage:**
1. Run **Cell 1** to install dependencies.
2. Run **Cell 2** to set the output folder name for this subject.
3. Run **Cell 3** to upload your PDF(s) and build the index.
4. Run **Cell 4** to download the generated folder as a `.zip`.
5. Unzip and place the folder next to `ibt.py` in your project.

In [ ]:
# ─── Cell 1: Install Dependencies ─────────────────────────────────────────────
import sys
print(f"🐍 Python {sys.version}\n")

packages = [
    'pypdf',                    # Modern PDF reader (replaces deprecated PyPDF2)
    'langchain',
    'langchain-community',
    'langchain-text-splitters',
    'langchain-huggingface',    # IMPORTANT: must match the Streamlit app
    'sentence-transformers',
    'faiss-cpu',
]

for pkg in packages:
    print(f"📥 Installing {pkg}...")
    result = !pip install -q {pkg}
    print(f"   ✅ done")

print("\n✅ All packages installed!")

In [ ]:
# ─── Cell 2: Configure Subject ─────────────────────────────────────────────────
# Set this to match the 'faiss_dir' value in subjects_config.py
# Examples:
#   'data_mining_faiss'
#   'econometric_methods_faiss'
#   'mis_faiss'

OUTPUT_NAME = 'data_mining_faiss'   # ← CHANGE THIS FOR EACH SUBJECT

print(f"📁 Output folder will be: {OUTPUT_NAME}/")

In [ ]:
# ─── Cell 3: Upload PDFs → Build FAISS Index with Metadata ─────────────────────
import os
import zipfile
from pypdf import PdfReader
from google.colab import files
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# IMPORTANT: Use langchain_huggingface (same package as the Streamlit app)
# This prevents pickle deserialization errors caused by module path mismatches.
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ── Step 1: Upload ──────────────────────────────────────────────────────────
print("📁 Please upload one or more PDF files for this subject:")
uploaded = files.upload()

pdf_files = [f for f in uploaded.keys() if f.lower().endswith('.pdf')]
if not pdf_files:
    raise ValueError("❌ No PDF files uploaded. Please upload at least one .pdf file.")

print(f"\n✅ {len(pdf_files)} PDF(s) uploaded:")
for pdf in pdf_files:
    size_mb = os.path.getsize(pdf) / (1024 * 1024)
    print(f"   📄 {pdf}  ({size_mb:.2f} MB)")

# ── Step 2: Extract Text with Page Metadata ─────────────────────────────────
print("\n" + "="*60)
print("📖 EXTRACTING TEXT & METADATA FROM PDF(s)")
print("="*60)

raw_docs = []
total_pages = 0

for pdf_file in pdf_files:
    print(f"\n  Processing: {pdf_file}")
    reader = PdfReader(pdf_file)
    pages = len(reader.pages)
    total_pages += pages
    print(f"  📄 Pages: {pages}")
    file_chars = 0
    for i, page in enumerate(reader.pages):
        page_text = page.extract_text()
        if page_text and page_text.strip():
            file_chars += len(page_text)
            raw_docs.append(Document(
                page_content=page_text,
                metadata={"source": pdf_file, "page": i}
            ))
        if (i + 1) % 10 == 0:
            print(f"     Progress: {i+1}/{pages} pages ({(i+1)/pages*100:.0f}%)")
    print(f"  ✅ Extracted {file_chars:,} characters")

print(f"\n📝 Total extracted pages: {len(raw_docs)} across {total_pages} pages")

if not raw_docs:
    print("⚠️  WARNING: Very little text extracted. The PDF(s) may be scanned images.")
    print("   Consider using an OCR tool before uploading.")

# ── Step 3: Chunk Documents with Preserved Metadata ───────────────────────
print("\n" + "="*60)
print("✂️  SPLITTING TEXT INTO CHUNKS (Preserving Source & Page Metadata)")
print("="*60)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
)
chunks = splitter.split_documents(raw_docs)
print(f"✅ Created {len(chunks):,} text chunks with metadata")
if chunks:
    print(f"\n📝 Sample chunk (first 200 chars):")
    print(f"   {chunks[0].page_content[:200]}...")
    print(f"   Metadata: {chunks[0].metadata}")

# ── Step 4: Build Embeddings ────────────────────────────────────────────────
print("\n" + "="*60)
print("🧠 CREATING EMBEDDINGS")
print("="*60)
print("⏳ This may take 5–20 minutes depending on document size…\n")

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
print(f"📥 Loading model: {MODEL_NAME}")

embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={"device": "cpu"},
)
print("✅ Embedding model loaded")

print(f"\n🔢 Encoding {len(chunks)} chunks into FAISS index…")
vector_store = FAISS.from_documents(chunks, embedding=embeddings)
print("✅ FAISS index created!")

# ── Step 5: Save with FAISS Native Format ───────────────────────────────────
# IMPORTANT: Use save_local/load_local — NOT pickle.dump.
# This is stable across langchain versions and avoids module-path mismatch errors.
print("\n" + "="*60)
print("💾 SAVING VECTOR STORE")
print("="*60)

os.makedirs(OUTPUT_NAME, exist_ok=True)
vector_store.save_local(OUTPUT_NAME)

saved_files = os.listdir(OUTPUT_NAME)
print(f"✅ FAISS index saved to folder: {OUTPUT_NAME}/")
print(f"   Files: {saved_files}")

total_size = sum(
    os.path.getsize(os.path.join(OUTPUT_NAME, f))
    for f in saved_files
) / (1024 * 1024)
print(f"   Total size: {total_size:.2f} MB")

# ── Step 6: Quick Sanity Test ───────────────────────────────────────────────
print("\n" + "="*60)
print("🧪 TESTING VECTOR STORE")
print("="*60)

test_store = FAISS.load_local(
    OUTPUT_NAME, embeddings, allow_dangerous_deserialization=True
)
test_results = test_store.similarity_search("introduction", k=3)
print(f"✅ Test passed! Found {len(test_results)} documents for test query.")
print(f"   Preview: {test_results[0].page_content[:150]}...")
print(f"   Metadata preview: {test_results[0].metadata}")

print("\n🎉 Vector store is ready!  Now run Cell 4 to download.")

In [ ]:
# ─── Cell 4: Package and Download ─────────────────────────────────────────────
import zipfile
from google.colab import files

zip_name = f"{OUTPUT_NAME}.zip"

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_NAME):
        zf.write(os.path.join(OUTPUT_NAME, fname), arcname=os.path.join(OUTPUT_NAME, fname))

zip_size_mb = os.path.getsize(zip_name) / (1024 * 1024)
print(f"📦 Created archive: {zip_name}  ({zip_size_mb:.2f} MB)")
print()
print("⬇️  Downloading…")
files.download(zip_name)

print("\n" + "="*60)
print("📋 NEXT STEPS")
print("="*60)
print(f"  1. Check your Downloads folder for: {zip_name}")
print(f"  2. Unzip it — you'll get a folder called: {OUTPUT_NAME}/")
print(f"  3. Place that folder next to ibt.py in your project")
print(f"  4. Make sure subjects_config.py has 'faiss_dir': '{OUTPUT_NAME}'")
print(f"  5. Push to GitHub and Streamlit Cloud will auto-deploy!")
print("="*60)